## 0: Setup

In [1]:
import os
import requests
import time
import json

from dotenv import load_dotenv
from google import genai
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from bs4 import BeautifulSoup
from openai import OpenAI

In [2]:
load_dotenv()

GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
client = genai.Client(api_key=GROQ_API_KEY)

In [3]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

def llm_generate(prompt: str) -> str:
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1000
    )
    return response.choices[0].message.content

## 1: Embedding

In [4]:
client = genai.Client(api_key=GEMINI_API_KEY)

result = client.models.embed_content(
    model="gemini-embedding-2",
    contents="embedding"
)
embedding = result.embeddings[0].values
print(f"Embedding dim: {len(embedding)}")

Embedding dim: 3072


In [5]:
qdrant = QdrantClient(path="./qdrant_storage")
qdrant.create_collection(
    collection_name="thailand_provinces",
    vectors_config=VectorParams(
        size=3072,  
        distance=Distance.COSINE
    )
)
print("✅ Qdrant ready")
print(f"Collections: {qdrant.get_collections()}")

✅ Qdrant ready
Collections: collections=[CollectionDescription(name='thailand_provinces')]


## 2: Generate Strucutred Data

In [6]:
import json
import time

def generate_category(category: str, prompt: str) -> any:
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=2000
    )
    raw = response.choices[0].message.content.strip()
    clean = raw.replace("```json", "").replace("```", "").strip()
    try:
        return json.loads(clean)
    except Exception as e:
        print(f"❌ Parse error {category}: {e}")
        print(f"Raw: {raw[:200]}")
        return []

pantip_context = """
- สวนอินทผลัม, เขื่อนขุนด่านปราการชล
- คาเฟ่ภูตะลึง, อ่างเก็บน้ำห้วยปรือ, วัดมณีวงศ์
- ATV หลายเจ้า มีปัญหาด้านรอคิว
- เขาช่องลม มีที่พักใกล้เคียง
- น้ำตก มีของกินอร่อยแวะได้
- เขื่อนขุนด่านปล่อยน้ำผ่าน spillways
"""

categories = {
    "places": f"""
Generate TOP 8 attractions in Nakhon Nayok for foreign tourists.
Based on: {pantip_context}

Return a JSON ARRAY (not object) of places. Each item must have:
- name: {{english, thai}}
- description
- best_time_to_visit
- entry_fee: {{thb, usd}}
- tips_for_foreigners
- hidden_gem_score (1-10)

IMPORTANT: Return ONLY a JSON array starting with [ and ending with ]
No markdown, no backticks, no extra text.
""",

    "transport": f"""
Generate transport guide from Bangkok to Nakhon Nayok.

Return a JSON OBJECT with keys:
- transport_options_from_bangkok: array of options
- local_transport_within_nakhon_nayok: array of local transport
- tips_without_thai_language_skills: array of tips

Each transport option must have:
- option_name, description, duration
- cost_thb, cost_usd
- departure_points_bangkok: [{{name, location}}]
- notes

IMPORTANT: Return ONLY a JSON object starting with {{ and ending with }}
No markdown, no backticks, no extra text.
""",

    "food": f"""
Generate 4 must-try foods in Nakhon Nayok for foreign tourists.

Return a JSON ARRAY. Each item must have:
- dish_name, description
- restaurant_name, restaurant_type, location
- price_range_thb, price_range_usd
- spicy_level, vegetarian_friendly

IMPORTANT: Return ONLY a JSON array starting with [ and ending with ]
No markdown, no backticks, no extra text.
""",

    "activities": f"""
Generate 5 activities in Nakhon Nayok for foreign tourists.
Based on: {pantip_context}

Return a JSON ARRAY. Each item must have:
- activity_name, description
- duration, cost_thb, cost_usd
- best_season, difficulty_level, booking_tips

IMPORTANT: Return ONLY a JSON array starting with [ and ending with ]
No markdown, no backticks, no extra text.
""",

    "itinerary": f"""
Generate 3 itineraries for Nakhon Nayok:
1. One Day Trip from Bangkok
2. Weekend Trip (2 days 1 night)
3. Extended Trip (3 days 2 nights)

Return a JSON OBJECT with key:
- nakhon_nayok_itineraries: array of itineraries

Each itinerary must have:
- trip_type, schedule, best_season
- estimated_budget_per_person: {{THB, USD}}
- transport_details: {{from_bangkok, local_transport}}
- accommodation_suggestions

IMPORTANT: Return ONLY a JSON object starting with {{ and ending with }}
No markdown, no backticks, no extra text.
"""
}

# Generate ทีละ category
knowledge_base = {}

for category, prompt in categories.items():
    print(f"⏳ Generating {category}...")
    knowledge_base[category] = generate_category(category, prompt)
    print(f"✅ {category}: {type(knowledge_base[category]).__name__}")
    time.sleep(3)  # ป้องกัน rate limit

# Save
with open("nakhonnayok_knowledge_base.json", "w", encoding="utf-8") as f:
    json.dump(knowledge_base, f, ensure_ascii=False, indent=2)

print("\n✅ Knowledge base saved!")

⏳ Generating places...
✅ places: list
⏳ Generating transport...
✅ transport: dict
⏳ Generating food...
✅ food: list
⏳ Generating activities...
✅ activities: list
⏳ Generating itinerary...
✅ itinerary: dict

✅ Knowledge base saved!


## 3: Chunking

In [7]:
# Cell 8: แปลง knowledge base เป็น chunks
import uuid
from qdrant_client.models import PointStruct

with open("nakhonnayok_knowledge_base.json", "r", encoding="utf-8") as f:
    raw = json.load(f)

def clean_kb(kb: dict) -> dict:
    cleaned = {}
    for key, value in kb.items():
        if isinstance(value, str):
            clean = value.strip().replace("```json", "").replace("```", "").strip()
            try:
                cleaned[key] = json.loads(clean)
            except:
                print(f"❌ Cannot parse {key}")
                cleaned[key] = value
        else:
            cleaned[key] = value
    return cleaned

kb = clean_kb(raw)

def safe_get(obj, *keys, default="N/A"):
    """ดึง nested key อย่างปลอดภัย"""
    for key in keys:
        if isinstance(obj, dict):
            obj = obj.get(key, default)
        else:
            return default
    return obj

def flatten_to_chunks(kb: dict) -> list[dict]:
    chunks = []

    # Places
    for place in kb.get("places", []):
        if not isinstance(place, dict):
            continue
        text = f"""
Place: {safe_get(place, 'name', 'english')} ({safe_get(place, 'name', 'thai')})
Description: {place.get('description', 'N/A')}
Best time to visit: {place.get('best_time_to_visit', 'N/A')}
Entry fee: {safe_get(place, 'entry_fee', 'thb')} THB / {safe_get(place, 'entry_fee', 'usd')} USD
Tips: {place.get('tips_for_foreigners', 'N/A')}
Hidden gem score: {place.get('hidden_gem_score', 'N/A')}/10
        """.strip()
        chunks.append({"category": "place", "text": text})

    # Transport
    transport = kb.get("transport", {})
    if isinstance(transport, dict):
        for option in transport.get("transport_options_from_bangkok", []):
            if not isinstance(option, dict):
                continue
            departure = "N/A"
            dep_list = option.get("departure_points_bangkok", [])
            if dep_list and isinstance(dep_list[0], dict):
                departure = dep_list[0].get("name", "N/A")
            elif dep_list and isinstance(dep_list[0], str):
                departure = dep_list[0]

            text = f"""
Transport option: {option.get('option_name', 'N/A')} from Bangkok to Nakhon Nayok
Description: {option.get('description', 'N/A')}
Duration: {option.get('duration', 'N/A')}
Cost: {option.get('cost_thb', 'N/A')} THB / {option.get('cost_usd', 'N/A')} USD
Departure point: {departure}
Notes: {option.get('notes', 'N/A')}
            """.strip()
            chunks.append({"category": "transport", "text": text})

        for local in transport.get("local_transport_within_nakhon_nayok", []):
            if isinstance(local, str):
                text = f"Local transport in Nakhon Nayok: {local}"
            elif isinstance(local, dict):
                text = f"""
Local transport in Nakhon Nayok: {local.get('transport_type', 'N/A')}
Description: {local.get('description', 'N/A')}
Cost: {local.get('cost_thb', 'N/A')}
Tips: {local.get('tips', 'N/A')}
                """.strip()
            else:
                continue
            chunks.append({"category": "transport_local", "text": text})

    # Food
    for food in kb.get("food", []):
        if not isinstance(food, dict):
            continue
        text = f"""
Food: {food.get('dish_name', 'N/A')}
Description: {food.get('description', 'N/A')}
Restaurant: {food.get('restaurant_name', 'N/A')} ({food.get('restaurant_type', 'N/A')})
Location: {food.get('location', 'N/A')}
Price: {food.get('price_range_thb', 'N/A')} THB / {food.get('price_range_usd', 'N/A')} USD
Spicy level: {food.get('spicy_level', 'N/A')}
Vegetarian friendly: {food.get('vegetarian_friendly', 'N/A')}
        """.strip()
        chunks.append({"category": "food", "text": text})

    # Activities
    for act in kb.get("activities", []):
        if not isinstance(act, dict):
            continue
        text = f"""
Activity: {act.get('activity_name', 'N/A')}
Description: {act.get('description', 'N/A')}
Duration: {act.get('duration', 'N/A')}
Cost: {act.get('cost_thb', 'N/A')} THB / {act.get('cost_usd', 'N/A')} USD
Best season: {act.get('best_season', 'N/A')}
Difficulty: {act.get('difficulty_level', 'N/A')}
Booking tips: {act.get('booking_tips', 'N/A')}
        """.strip()
        chunks.append({"category": "activity", "text": text})

    # Itinerary
    itinerary = kb.get("itinerary", {})
    if isinstance(itinerary, list):
        itineraries = itinerary
    elif isinstance(itinerary, dict):
        itineraries = itinerary.get("nakhon_nayok_itineraries", [])
    else:
        itineraries = []

    for itin in itineraries:
        if not isinstance(itin, dict):
            continue
        budget = itin.get("estimated_budget_per_person", {})
        transport_details = itin.get("transport_details", {})
        text = f"""
Itinerary type: {itin.get('trip_type', 'N/A')}
Budget per person: {budget.get('THB', 'N/A')} THB / {budget.get('USD', 'N/A')} USD
Best season: {itin.get('best_season', 'N/A')}
Transport from Bangkok: {transport_details.get('from_bangkok', 'N/A')}
        """.strip()
        chunks.append({"category": "itinerary", "text": text})

    return chunks

chunks = flatten_to_chunks(kb)
print(f"✅ Total chunks: {len(chunks)}")
for c in chunks:
    print(f"  [{c['category']}] {c['text'][:60]}...")

✅ Total chunks: 27
  [place] Place: Khun Dan Prakan Chon Dam (เขื่อนขุนด่านปราการชล)
Desc...
  [place] Place: Phu Talung Cafe (คาเฟ่ภูตะลึง)
Description: A cafe wi...
  [place] Place: Wang Ta Krai Waterfall (น้ำตก)
Description: A beautif...
  [place] Place: Khao Chong Lom (เขาช่องลม)
Description: A hill with a...
  [place] Place: Huai Prue Reservoir (อ่างเก็บน้ำห้วยปรือ)
Description...
  [place] Place: Manee Wong Temple (วัดมณีวงศ์)
Description: A histori...
  [place] Place: Many ATV (ATV หลายเจ้า)
Description: An ATV track wit...
  [place] Place: Fig Garden (สวนอินทผลัม)
Description: A beautiful gar...
  [transport] Transport option: Minivan from Bangkok to Nakhon Nayok
Descr...
  [transport] Transport option: Bus from Bangkok to Nakhon Nayok
Descripti...
  [transport] Transport option: Taxi from Bangkok to Nakhon Nayok
Descript...
  [transport_local] Local transport in Nakhon Nayok: Songthaew (red truck)...
  [transport_local] Local transport in Nakhon Nayok: Tuk-tuk...
  [transport_

In [8]:
points = []

for i, chunk in enumerate(chunks):
    result = client.models.embed_content(
        model="gemini-embedding-2",
        contents=chunk["text"]
    )
    vector = result.embeddings[0].values

    points.append(PointStruct(
        id=str(uuid.uuid4()),
        vector=vector,
        payload={
            "category": chunk["category"],
            "text": chunk["text"]
        }
    ))
    print(f"✅ {i+1}/{len(chunks)} [{chunk['category']}]")

# Upload ทั้งหมด
qdrant.upsert(
    collection_name="thailand_provinces",
    points=points
)

print(f"\n✅ Uploaded {len(points)} vectors to Qdrant")

✅ 1/27 [place]
✅ 2/27 [place]
✅ 3/27 [place]
✅ 4/27 [place]
✅ 5/27 [place]
✅ 6/27 [place]
✅ 7/27 [place]
✅ 8/27 [place]
✅ 9/27 [transport]
✅ 10/27 [transport]
✅ 11/27 [transport]
✅ 12/27 [transport_local]
✅ 13/27 [transport_local]
✅ 14/27 [transport_local]
✅ 15/27 [transport_local]
✅ 16/27 [food]
✅ 17/27 [food]
✅ 18/27 [food]
✅ 19/27 [food]
✅ 20/27 [activity]
✅ 21/27 [activity]
✅ 22/27 [activity]
✅ 23/27 [activity]
✅ 24/27 [activity]
✅ 25/27 [itinerary]
✅ 26/27 [itinerary]
✅ 27/27 [itinerary]

✅ Uploaded 27 vectors to Qdrant


## 4: RAG query

In [9]:
def retrieve(query: str, top_k: int = 5) -> list[dict]:
    result = client.models.embed_content(
        model="gemini-embedding-2",
        contents=query
    )
    hits = qdrant.query_points(
        collection_name="thailand_provinces",
        query=result.embeddings[0].values,
        limit=top_k
    ).points
    return [{"score": h.score, "category": h.payload["category"], "text": h.payload["text"]} for h in hits]


In [10]:
def llm_generate(prompt: str) -> str:
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1000
    )
    return response.choices[0].message.content

In [11]:
def rag_search(query: str) -> str:
    results = retrieve(query, top_k=5)
    return "\n\n---\n\n".join([r["text"] for r in results]) if results else "No results."

def google_search(query: str) -> str:
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=f"Search and answer with current info: {query} Nakhon Nayok Thailand",
        config={"tools": [{"google_search": {}}]}
    )
    return response.text

def build_itinerary(days: int, budget_usd: float, interests: str) -> str:
    context = rag_search(f"{days} day itinerary {interests}")
    return llm_generate(f"Context: {context}\n\nBuild {days}-day itinerary, budget ${budget_usd}/person, interests: {interests}. Include times, costs in THB+USD.")

tool_map = {
    "rag_search":      lambda args: rag_search(args["query"]),
    "google_search":   lambda args: google_search(args["query"]),
    "build_itinerary": lambda args: build_itinerary(args["days"], args["budget_usd"], args["interests"])
}

In [12]:
SYSTEM = """You are a friendly travel companion for Nakhon Nayok, Thailand.
Help foreign tourists plan their trip with practical, honest advice.
Always include prices in both THB and USD."""

def agent_chat(user_message: str) -> str:
    context = rag_search(user_message)
    prompt = f"{SYSTEM}\n\nCONTEXT:\n{context}\n\nUSER: {user_message}\nASSISTANT:"
    return llm_generate(prompt)

print("✅ Agent ready")


✅ Agent ready


In [13]:
print(agent_chat("What are the best places to visit in Nakhon Nayok?"))


Nakhon Nayok is a beautiful province with a lot to offer. I'd highly recommend visiting Khun Dan Prakan Chon Dam (เขื่อนขุนด่านปราการชล) during the rainy season when the dam is full, the scenery is just stunning, especially when water is released through the spillways. And the best part? It's completely free to visit, so you can enjoy the views without spending a single THB (or USD).

Another must-visit is the Wang Ta Krai Waterfall (น้ำตก), which is at its peak after rainfall. The entry fee is only 20 THB / 0.6 USD, and you can enjoy the beautiful waterfall, natural pool, and try the delicious local cuisine at the surrounding restaurants.

If you're looking for a mix of nature and culture, I'd suggest the Dam and Temple Tour, which visits Khun Dan Prakan Chon Dam and the nearby วัดมณีวงศ์ temple. The tour costs 200 THB / 6 USD and lasts for 4 hours, and it's available all year round.

If you have time, you could also visit a fruit garden, such as สวนอินทผลัม, which offers a beautiful 